In [1]:
from langchain_huggingface import HuggingFaceEndpointEmbeddings
from langchain_chroma import Chroma

c:\Users\iveda\OneDrive\Desktop\gen_ai\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
from langchain_core.documents import Document

# Create LangChain documents for IPL players

doc1 = Document(
        page_content="Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.",
        metadata={"team": "Royal Challengers Bangalore"}
    )
doc2 = Document(
        page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
        metadata={"team": "Mumbai Indians"}
    )
doc3 = Document(
        page_content="MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.",
        metadata={"team": "Chennai Super Kings"}
    )
doc4 = Document(
        page_content="Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.",
        metadata={"team": "Mumbai Indians"}
    )
doc5 = Document(
        page_content="Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.",
        metadata={"team": "Chennai Super Kings"}
    )

In [6]:
docs = [doc1, doc2, doc3, doc4, doc5]

In [11]:
# Embedding Function
from langchain_huggingface import HuggingFaceEndpointEmbeddings
import os 
from dotenv import load_dotenv

load_dotenv()
embeddings = HuggingFaceEndpointEmbeddings(
    # model="BAAI/bge-large-en-v1.5", #1024
    model="BAAI/bge-base-en-v1.5",  #768
    huggingfacehub_api_token=os.getenv("HUGGINGFACEHUB_API_TOKEN")
)

In [ ]:
# vector = embeddings.embed_query("What is LangChain")
# print(len(vector))

768


In [14]:
vector_store = Chroma(
    embedding_function=embeddings,
    persist_directory='chroma_db',
    collection_name='sample'
)

In [15]:
# add dcouments
vector_store.add_documents(docs)

['be83b3fa-8d10-4b2a-93ae-e54b687840cb',
 '567b3d1a-47bf-4793-ab31-794d22c8d84b',
 '62b5f991-a801-4d67-a74f-568720ad2c6f',
 'e4eb2586-7cdd-45aa-b976-790ec3f00178',
 'c2b25aa8-42e8-4fdd-97bd-c6b59ee0913c']

In [16]:
vector_store.get(include=["embeddings", 'documents', 'metadatas'])

{'ids': ['be83b3fa-8d10-4b2a-93ae-e54b687840cb',
  '567b3d1a-47bf-4793-ab31-794d22c8d84b',
  '62b5f991-a801-4d67-a74f-568720ad2c6f',
  'e4eb2586-7cdd-45aa-b976-790ec3f00178',
  'c2b25aa8-42e8-4fdd-97bd-c6b59ee0913c'],
 'embeddings': array([[-0.02164437, -0.05776449, -0.00329511, ..., -0.00129844,
         -0.01330102, -0.04032261],
        [-0.08374138, -0.07472931,  0.00594075, ...,  0.02139488,
          0.01442805, -0.08096205],
        [-0.04401276, -0.04386465,  0.01327865, ..., -0.00979984,
          0.00444421, -0.01370721],
        [-0.06514502, -0.0873289 , -0.01727353, ...,  0.04433516,
          0.04437803, -0.04213695],
        [-0.01186087, -0.02604065, -0.04370598, ...,  0.01963787,
          0.01333483, -0.04212806]], shape=(5, 768)),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the mo

In [17]:
vector_store.similarity_search(
    query="Who among these are a bowler?",
    k=2
)

[Document(id='e4eb2586-7cdd-45aa-b976-790ec3f00178', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
 Document(id='567b3d1a-47bf-4793-ab31-794d22c8d84b', metadata={'team': 'Mumbai Indians'}, page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.")]

In [19]:
vector_store.similarity_search_with_score(
    query="Who among these are a bowler?",
    k=2
)

[(Document(id='e4eb2586-7cdd-45aa-b976-790ec3f00178', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
  0.73823082447052),
 (Document(id='567b3d1a-47bf-4793-ab31-794d22c8d84b', metadata={'team': 'Mumbai Indians'}, page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure."),
  0.9056082963943481)]

In [20]:
vector_store.similarity_search_with_score(
    query="",
    filter={'team': 'Chennai Super Kings'}
)

[(Document(id='62b5f991-a801-4d67-a74f-568720ad2c6f', metadata={'team': 'Chennai Super Kings'}, page_content='MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.'),
  1.0925500392913818),
 (Document(id='c2b25aa8-42e8-4fdd-97bd-c6b59ee0913c', metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
  1.1679966449737549)]

In [21]:
# update documents
updated_doc1 = Document(
    page_content="Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple centuries in a single season. Despite RCB not winning an IPL title under his captaincy, Kohli's passion and fitness set a benchmark for the league. His ability to chase targets and anchor innings has made him one of the most dependable players in T20 cricket.",
    metadata={"team": "Royal Challengers Bangalore"}
)

vector_store.update_document(document_id='be83b3fa-8d10-4b2a-93ae-e54b687840cb', document=updated_doc1)

In [22]:
vector_store.get(include=["embeddings", 'documents', 'metadatas'])

{'ids': ['be83b3fa-8d10-4b2a-93ae-e54b687840cb',
  '567b3d1a-47bf-4793-ab31-794d22c8d84b',
  '62b5f991-a801-4d67-a74f-568720ad2c6f',
  'e4eb2586-7cdd-45aa-b976-790ec3f00178',
  'c2b25aa8-42e8-4fdd-97bd-c6b59ee0913c'],
 'embeddings': array([[-0.04695304, -0.07157257, -0.02110945, ...,  0.02568371,
          0.00359523, -0.03638843],
        [-0.08374138, -0.07472931,  0.00594075, ...,  0.02139488,
          0.01442805, -0.08096205],
        [-0.04401276, -0.04386465,  0.01327865, ..., -0.00979984,
          0.00444421, -0.01370721],
        [-0.06514502, -0.0873289 , -0.01727353, ...,  0.04433516,
          0.04437803, -0.04213695],
        [-0.01186087, -0.02604065, -0.04370598, ...,  0.01963787,
          0.01333483, -0.04212806]], shape=(5, 768)),
 'documents': ["Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple ce